# Track C Kaggle vLLM handoff

Enable GPU T4 ×2 and Internet in Kaggle settings. Attach both the source archive and the frozen v7 data archive supplied with `docs/track_c_kaggle_handoff.md`. Their checksums are recorded separately. Model weights are downloaded from pinned public revisions.

This notebook verifies transferred data, waits for vLLM readiness, runs and audits the declared conditions, checks unchanged resume, and records visible allocated GPU-hours even if a run fails. Saved C1/C2 notebooks provide earlier execution evidence; this handoff starts with blank outputs and makes no claim that it has already run.


In [ ]:
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/lma-grounding-decay")
source_roots = sorted(INPUT.rglob("pyproject.toml"))
assert len(source_roots) == 1, "Attach the supplied source archive as one Kaggle dataset"
shutil.copytree(source_roots[0].parent, PROJECT, dirs_exist_ok=True)
asset_manifests = sorted(INPUT.rglob("bundle_manifest.json"))
assert len(asset_manifests) == 1, "Attach the supplied frozen data archive"
shutil.copytree(asset_manifests[0].parent / "data", PROJECT / "data", dirs_exist_ok=True)
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT / "src")
print("Project and frozen data copied to:", PROJECT)

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        ".[dev]",
        "vllm==0.32.0",
        "sentence-transformers==6.2.0",
        "faiss-cpu==1.15.1",
    ],
    check=True,
)
# After dependency installation, restart the kernel if Kaggle requests it, then resume here.

In [ ]:
from huggingface_hub import snapshot_download

for model, revision in [
    ("BAAI/bge-m3", "5617a9f61b028005a4858fdac845db406aefb181"),
    ("BAAI/bge-reranker-v2-m3", "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"),
]:
    snapshot_download(repo_id=model, revision=revision)

In [ ]:
import json

selection = json.loads(Path("configs/model_selection.json").read_text(encoding="utf-8"))
model = os.environ.get("LMA_VLLM_MODEL_ID", selection["vllm"]["model"])
revision = os.environ.get("LMA_VLLM_MODEL_REVISION", selection["vllm"]["revision"])
command = [
    sys.executable,
    "scripts/kaggle_run.py",
    "--model",
    model,
    "--revision",
    revision,
    "--conditions",
    "B0",
    "B1",
    "B1-R",
    "B2",
    "A-NoVerify",
    "A-NoLoop",
    "M",
]
subprocess.run([*command, "--verify-only"], check=True, env=os.environ.copy())

In [ ]:
subprocess.run(command, check=True, env=os.environ.copy())

In [ ]:
output = Path("/kaggle/working/track_c_run")
print((output / "audit.json").read_text(encoding="utf-8"))
for path in sorted(output.glob("compute-*.json")):
    print(path.name, path.read_text(encoding="utf-8"))